### ADCP Data

* This notebook is to treat the raw data from the two moored ADCP in Faro inlet

    * ADCP1: Olhao Channel
    * ADCP2: Faro Channel

---
`MoeinDst, version 1.0, 11.07.24, Faro, Portugal`

In [ ]:
import numpy as np
import scipy.io
import pandas as pd
import os, sys

In [ ]:
sys.path.append('/home/moein/myROMS/general_notebooks/PyFuncs/')
import PostProcess_Functions as ppfs

In [ ]:
adcp1 = scipy.io.loadmat('../Data/ADCP1.mat')
adcp2 = scipy.io.loadmat('../Data/ADCP2.mat')

Extracting data from `.mat` files as dictionary.

In [ ]:
###---- ADCP1 ----###
keyNames1 = list(adcp1.keys())

sens1 = dict()
for var in ['time', 'h', 'p', 'r', 'o', 'v', 't', 'pd', 'sos', 'dnum']:
    sens1[var] = np.concatenate(list(adcp1['sens'][var][0,0]))
time1 = sens1['time']
sens1['time'] = pd.to_datetime(sens1['time'], unit='s', origin='unix')


vb1 = dict()
for var in ['vel', 'int', 'cor']:
    vb1[var] = (adcp1['vb'][var][0,0])

wt1 = dict()
for var in ['vel', 'int', 'corr', 'pg', 'd', 'r']:
    wt1[var] = (adcp1['wt'][var][0,0])

In [ ]:
###---- ADCP2 ----###
keyNames2 = list(adcp2.keys())

sens2 = dict()
for var in ['time', 'h', 'p', 'r', 'o', 'v', 't', 'pd', 'sos', 'dnum']:
    sens2[var] = np.concatenate(list(adcp2['sens'][var][0,0]))
time2 = sens2['time']
sens2['time'] = pd.to_datetime(sens2['time'], unit='s', origin='unix')

vb2 = dict()
for var in ['vel', 'int', 'cor']:
    vb2[var] = (adcp2['vb'][var][0,0])

wt2 = dict()
for var in ['vel', 'int', 'corr', 'pg', 'd', 'r']:
    wt2[var] = (adcp2['wt'][var][0,0])

The four dimensions in wt['vel'] refer to EastWard-Velocity, NorthWard-Velocity, Up-component, and error, respectively.

---
Morover, velocity data is Eastward and Northward. It is desirableto rotate them to the main axis and have them as along-channel and cross-channel! Therefore, we use principal axes of variance/current flow, and then rotate them. Below, first we apply the quality control and filter the data, and then do the PCA.

* The common practice is to use the `pg` being percent good to do the quality check on the records together with the value of error recorded automatically by the instrument. Here, the way that the `pg` was written by the instrument's software was not really clear, so we used the correlation records of different beams and the intensity threshold.

In [ ]:
# Quality Check
intens_thresh = 70 # intensity
corr_thresh = 60 # correlation

quality_filter1 = (wt1['int'] < intens_thresh) | (wt1['corr'] < corr_thresh)
quality_filter2 = (wt2['int'] < intens_thresh) | (wt2['corr'] < corr_thresh)

vel1_filtered = np.where(quality_filter1, np.nan, wt1['vel'])
u1_filtered = vel1_filtered[:,:,0]
v1_filtered = vel1_filtered[:,:,1]

vel2_filtered = np.where(quality_filter2, np.nan, wt2['vel'])
u2_filtered = vel2_filtered[:,:,0]
v2_filtered = vel2_filtered[:,:,1]

In [ ]:
uclean_olhao = u1_filtered
vclean_olhao = v1_filtered

uclean_faro = u2_filtered
vclean_faro = v2_filtered

channels = {
    "olhao": (uclean_olhao, vclean_olhao),
    "faro": (uclean_faro, vclean_faro)
}

rotated_clean_velocities = {}

for channel_name, (u, v) in channels.items():
    num_depths = u.shape[1]

    # arrays to store results
    b = np.zeros(num_depths)
    theta = np.zeros(num_depths)
    theta_degrees = np.zeros(num_depths)
    theta_corrected = np.zeros(num_depths)
    ucleanR = np.zeros_like(u)
    vcleanR = np.zeros_like(v)

    for i in range(num_depths):
        # PCA-based calculation for each depth
        n = u.shape[0]
        b[i] = (n * np.nansum(u[:, i] * v[:, i]) - np.nansum(u[:, i]) * np.nansum(v[:, i])) / \
               (n * np.nansum(u[:, i] ** 2) - (np.nansum(u[:, i])) ** 2)
        theta[i] = np.arctan2(b[i], 1)  # Rotation angle in radians
        theta_degrees[i] = np.rad2deg(theta[i])  # Convert to degrees
        theta_corrected[i] = (np.pi / 2) - theta[i]  # Corrected angle for rotation

        # Rotate velocities
        if channel_name == "faro":
        # Invert rotation for Faro (clockwise rotation)
            ucleanR[:, i] = u[:, i] * np.cos(theta_corrected[i]) - v[:, i] * np.sin(theta_corrected[i])
            vcleanR[:, i] = u[:, i] * np.sin(theta_corrected[i]) + v[:, i] * np.cos(theta_corrected[i])
        elif channel_name == "olhao":
            ucleanR[:, i] = u[:, i] * np.cos(theta_corrected[i]) - v[:, i] * np.sin(theta_corrected[i])
            vcleanR[:, i] = u[:, i] * np.sin(theta_corrected[i]) + v[:, i] * np.cos(theta_corrected[i])
    

    rotated_clean_velocities[channel_name] = {
        "ucleanR": ucleanR,
        "vcleanR": vcleanR,
        "theta_degrees": theta_degrees,
        "theta_corrected": theta_corrected
    }

In [ ]:
makeplot = True
if makeplot == True:
    import matplotlib.pyplot as plt

In [ ]:
if makeplot == True:
    
    fig1,(ax1_1,ax1_2) = plt.subplots(
        1,2,figsize=(9,5), constrained_layout=True, sharey=True,sharex=True
                                    )

    # Faro Channel
    ax1_1.scatter(uclean_faro,vclean_faro,
                  label='alonge-channel velocity')
    ax1_1.scatter(rotated_clean_velocities['faro']['ucleanR'],
                   rotated_clean_velocities['faro']['vcleanR'],
                  label='rotated velocity')
    ax1_1.set_title('Faro Channel', fontsize=9)
    # -----------------------------------------
    # Olhao Channel
    ax1_2.scatter(uclean_olhao,vclean_olhao,
                   label='alonge-channel velocity')
    ax1_2.scatter(rotated_clean_velocities['olhao']['ucleanR'],
                   rotated_clean_velocities['olhao']['vcleanR'],
                   label='rotated velocity')
    ax1_2.set_title('Olhao Channel', fontsize=9)


    for ax in (ax1_1,ax1_2):
        ax.grid(alpha=0.5)
        ax.legend(fontsize=6)

    fig1.suptitle('Scatter Plots of the Velocities from Moored ADCPs')

#### Renaming recorded variables consistently to be more informative

In [ ]:
fig,(ax_1,ax_2) = plt.subplots(
    2,1,figsize=(10,6), constrained_layout=True, sharex=True, sharey=True,
                                )
bin_num = 0

ax_1.plot(sens1['time'],u1_filtered[:,bin_num], label = 'u-east')
ax_1.plot(sens1['time'],v1_filtered[:,bin_num], label = 'v-north')
ax_1.set_ylabel('velocity [m/s]')
ax_1.set_title('ADCP1')

ax_2.plot(sens2['time'],u2_filtered[:,bin_num], label = 'u-east')
ax_2.plot(sens2['time'],v2_filtered[:,bin_num], label = 'v-north')
ax_2.set_xlabel('Time [YYYY.MM.DD]')
ax_2.set_ylabel('velocity [m/s]')
ax_2.set_title('ADCP2')

for ax in (ax_1,ax_2):
    ax.grid(True,ls='-.',c='grey', alpha=0.2)
    ax.legend(fontsize=9)

bin_depth = wt2['r'][0,bin_num]
fig.suptitle(f'u-v velocitities at depth {bin_depth}m')

In [ ]:
binDepth1 = wt1['r'][0]
binDepth2 = wt2['r'][0]

In [ ]:
bDepth1, tTime1 = np.meshgrid(binDepth1,sens1['time'])
bDepth2, tTime2 = np.meshgrid(binDepth2,sens2['time'])

In [ ]:
water_depth1 = sens1['pd'] * 0.9
water_depth2 = sens2['pd'] * 0.9
### times 0.9 based on visual check

water_depth1filtered = np.where(quality_filter1[:,0,0], np.nan, water_depth1)
water_depth2filtered = np.where(quality_filter2[:,0,0], np.nan, water_depth2)

###----- Water depth filtering -----###
depth_mask1 = np.zeros_like(u1_filtered, dtype=bool)
depth_mask2 = np.zeros_like(u2_filtered, dtype=bool)

###----- Create depth mask -----###
for t in range(water_depth1.shape[0]):
    depth_mask1[t, binDepth1 > water_depth1[t]] = True
    
u1_filtered[depth_mask1] = np.nan
v1_filtered[depth_mask1] = np.nan
vel_mag1 = np.sqrt(u1_filtered**2 + v1_filtered**2)

for t in range(water_depth2.shape[0]):
    depth_mask2[t, binDepth2 > water_depth2[t]] = True
    
u2_filtered[depth_mask2] = np.nan
v2_filtered[depth_mask2] = np.nan
vel_mag2 = np.sqrt(u2_filtered**2 + v2_filtered**2)

In [ ]:
if makeplot == True:

    fig,(ax_1,ax_2) = plt.subplots(2,1, figsize=(12,6), constrained_layout=True, sharex=True)
    bin_num = 0

    mappable_1 = ax_1.pcolormesh(tTime1, bDepth1, vel_mag1)
    ax_1.plot(tTime1,water_depth1, lw=0.5, c='k')
    ax_1.set_ylabel('Depth [m]')
    ax_1.set_title('ADCP1_Olhao-Ch')

    mappable_2 = ax_2.pcolormesh(tTime2, bDepth2, vel_mag2)
    ax_2.plot(tTime2,water_depth2, lw=0.5, c='k')
    ax_2.set_xlabel('Time [YYYY.MM.DD]')
    ax_2.set_ylabel('Depth [m]')
    ax_2.set_title('ADCP2_Faro-Ch')
    
    # shared colorbar limits
    vmin = min(np.nanmin(vel_mag1), np.nanmin(vel_mag2))
    vmax = max(np.nanmax(vel_mag1), np.nanmax(vel_mag2))
    
    cbar = fig.colorbar(
    mappable_1,
    ax=[ax_1, ax_2],
    location='right',
    pad=0.02,
    fraction=0.03,
    aspect=40
                        )

    for ax in (ax_1,ax_2):
        ax.grid(True, ls='-', c='k', alpha=0.1)


                        <-------------------- End of Document -------------------->